# 06 — Visualizações Finais e Comparativo Internacional
**Objetivo:** Consolidar os gráficos mais impactantes e comparar Goiânia com Chernobyl e Fukushima.

Análises:
- Dashboard resumo do acidente (painel único)
- Linha do tempo do acidente
- Comparativo internacional padronizado
- Exportação de todos os gráficos para o README

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/processed/vitimas_clean.csv")
locais = pd.read_csv("../data/processed/locais_clean.csv")

print("Dados carregados com sucesso.")
print(f"Vitimas: {len(df)} | Obitos: {df['obito'].sum()} | Internados: {df['internado'].sum()}")


## 6.1 — Painel Resumo do Acidente (Dashboard Estático)

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0D1117')

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

title_style = {'fontsize': 10, 'fontweight': 'bold', 'color': '#E6EDF3', 'pad': 8}
label_style = {'color': '#8B949E', 'fontsize': 8}

def dark_ax(ax):
    ax.set_facecolor('#161B22')
    ax.tick_params(colors='#8B949E', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363D')
    return ax

# ── KPIs ──────────────────────────────────────
kpis = [
    ('249', 'Pessoas
Contaminadas', '#58A6FF'),
    ('4',   'Obitos
Confirmados',  '#F85149'),
    ('49',  'Internados',           '#3FB950'),
    ('7',   'Pontos
Criticos',     '#D2A8FF'),
]
for i, (val, label, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    dark_ax(ax)
    ax.text(0.5, 0.58, val, ha='center', va='center', fontsize=32,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.18, label, ha='center', va='center', fontsize=9,
            color='#8B949E', transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])

# ── Distribuição de doses ──────────────────────
ax1 = dark_ax(fig.add_subplot(gs[1, :2]))
df['log_dose'] = np.log10(df['dose_mSv'] + 1)
ax1.hist(df['log_dose'], bins=35, color='#58A6FF', edgecolor='#0D1117', alpha=0.85)
ax1.axvline(x=np.log10(100), color='#F0883E', linestyle='--', linewidth=1.5, label='100 mSv')
ax1.axvline(x=np.log10(1000), color='#F85149', linestyle='--', linewidth=1.5, label='1000 mSv')
ax1.set_xlabel("log10(Dose em mSv)", **label_style)
ax1.set_ylabel("Frequencia", **label_style)
ax1.set_title("Distribuicao de Doses Recebidas", **title_style)
ax1.legend(fontsize=8, labelcolor='white', facecolor='#161B22', edgecolor='#30363D')

# ── Internação por grupo ───────────────────────
ax2 = dark_ax(fig.add_subplot(gs[1, 2:]))
grupos = df.groupby('grupo_exposicao')['internado'].agg(['sum', 'count'])
grupos['pct'] = grupos['sum'] / grupos['count'] * 100
order = ['leve', 'moderado', 'alto']
cores_g = ['#3FB950', '#F0883E', '#F85149']
bars = ax2.bar([g.capitalize() for g in order],
               [grupos.loc[g, 'pct'] for g in order],
               color=cores_g, edgecolor='#0D1117', linewidth=1.2, width=0.5)
for bar, g in zip(bars, order):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1.5,
             f"{grupos.loc[g, 'pct']:.0f}%",
             ha='center', fontsize=9, color='#E6EDF3', fontweight='bold')
ax2.set_ylabel("% Internados", **label_style)
ax2.set_title("Taxa de Internacao por Grupo de Exposicao", **title_style)
ax2.set_ylim(0, 115)

# ── Atividade por local ────────────────────────
ax3 = dark_ax(fig.add_subplot(gs[2, :2]))
cores_loc = {'critico': '#F85149', 'alto': '#F0883E', 'moderado': '#E3B341', 'baixo': '#3FB950'}
c_list = [cores_loc[n] for n in locais['nivel_contaminacao']]
ax3.barh(locais['bairro'], locais['atividade_GBq'], color=c_list, edgecolor='#0D1117')
ax3.set_xlabel("Atividade (GBq)", **label_style)
ax3.set_title("Atividade Radioativa por Ponto de Contaminacao", **title_style)

# ── Pessoas afetadas por bairro ────────────────
ax4 = dark_ax(fig.add_subplot(gs[2, 2:]))
bairro_df = df['bairro'].value_counts().reset_index()
bairro_df.columns = ['bairro', 'count']
ax4.bar(bairro_df['bairro'], bairro_df['count'],
        color='#D2A8FF', edgecolor='#0D1117', width=0.6)
ax4.set_ylabel("No de Pessoas", **label_style)
ax4.set_title("Pessoas Afetadas por Bairro", **title_style)
ax4.tick_params(axis='x', rotation=25)

fig.suptitle("☢  ACIDENTE RADIOLOGICO DE GOIANIA — 1987  |  ANALISE DE DADOS",
             fontsize=15, fontweight='bold', color='#E6EDF3', y=0.98)

plt.savefig("../reports/figures/06_dashboard_resumo.png", dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print("Dashboard salvo")


## 6.2 — Linha do Tempo do Acidente

In [ ]:
eventos = [
    ("13/09/1987", "Capsula encontrada
na clinica abandonada",       0),
    ("18/09/1987", "Devair compra o
aparelho no ferro-velho",        1),
    ("25/09/1987", "Cesio distribuido;
filha de Devair contaminada", 2),
    ("28/09/1987", "Primeiros sinais de
doenca em multiplas pessoas", 3),
    ("29/09/1987", "Vigilancia Sanitaria
identifica o acidente",      4),
    ("01/10/1987", "CNEN confirma
contaminacao por Cs-137",          5),
    ("23/10/1987", "1o obito —
Leide das Neves (6 anos)",            6),
    ("28/10/1987", "4o obito;
fim da fase aguda",                    7),
]

fig, ax = plt.subplots(figsize=(15, 5))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#0D1117')

ax.axhline(y=0, color='#30363D', linewidth=2.5, zorder=1)
for i, (data, texto, pos) in enumerate(eventos):
    y = 0.6 if i % 2 == 0 else -0.6
    color = '#F85149' if 'obito' in texto.lower() else '#58A6FF' if i == 0 else '#3FB950' if 'CNEN' in texto else '#E6EDF3'
    ax.scatter(i, 0, s=120, color=color, zorder=3, edgecolors='#0D1117', linewidths=2)
    ax.plot([i, i], [0, y * 0.85], color='#30363D', linewidth=1.2, zorder=2)
    ax.text(i, y, texto, ha='center', va='bottom' if y > 0 else 'top',
            fontsize=7.5, color='#E6EDF3', multialignment='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#161B22', edgecolor='#30363D', alpha=0.9))
    ax.text(i, -0.12 if y > 0 else 0.12, data, ha='center', va='top' if y > 0 else 'bottom',
            fontsize=7, color='#8B949E')

ax.set_xlim(-0.8, len(eventos) - 0.2)
ax.set_ylim(-1.3, 1.3)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("Linha do Tempo — Acidente Radiologico de Goiania (1987)",
             fontsize=13, fontweight='bold', color='#E6EDF3', pad=12)

plt.tight_layout()
plt.savefig("../reports/figures/06_linha_do_tempo.png", dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print("Linha do tempo salva")


## 6.3 — Comparativo Internacional: Goiânia × Chernobyl × Fukushima

In [ ]:
dados_comp = {
    'Acidente':           ['Goiania (1987)',    'Chernobyl (1986)',  'Fukushima (2011)'],
    'Tipo':               ['Radiologico',        'Nuclear',           'Nuclear'],
    'Contaminados_dir':   [249,                  600000,              170],
    'Obitos_agudos':      [4,                    31,                  1],
    'Area_km2':           [100,                  150000,              600],
    'Tempo_contencao_d':  [15,                   10,                  90],
    'Custo_bilhoes_USD':  [0.05,                 700,                 200],
    'Nivel_INES':         [5,                    7,                   7],
}
comp = pd.DataFrame(dados_comp)
print(comp.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor('#0D1117')
fig.suptitle("Comparativo Internacional de Acidentes Radiologicos",
             fontsize=14, fontweight='bold', color='#E6EDF3', y=1.01)

acidentes = comp['Acidente'].tolist()
cores_ac = ['#58A6FF', '#F85149', '#3FB950']

metricas = [
    ('Contaminados_dir', 'Pessoas Contaminadas', False),
    ('Obitos_agudos', 'Obitos Agudos', False),
    ('Area_km2', 'Area Contaminada (km²)', True),
    ('Tempo_contencao_d', 'Tempo de Contencao (dias)', False),
    ('Custo_bilhoes_USD', 'Custo Estimado (bi USD)', True),
    ('Nivel_INES', 'Nivel na Escala INES', False),
]

for ax, (col, titulo, log) in zip(axes.flat, metricas):
    ax.set_facecolor('#161B22')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363D')
    bars = ax.bar(acidentes, comp[col], color=cores_ac, edgecolor='#0D1117', width=0.55)
    for bar, val in zip(bars, comp[col]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * (1.08 if not log else 1.3),
                f'{val:,.0f}' if val >= 1 else f'{val:.2f}',
                ha='center', fontsize=8, color='#E6EDF3', fontweight='bold')
    if log:
        ax.set_yscale('log')
    ax.set_title(titulo, fontsize=9, fontweight='bold', color='#E6EDF3', pad=6)
    ax.tick_params(colors='#8B949E', labelsize=8)
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig("../reports/figures/06_comparativo_internacional.png", dpi=150,
            bbox_inches='tight', facecolor='#0D1117')
plt.show()
print("Comparativo internacional salvo")


## 6.4 — Exportação Final

In [ ]:
import os

figures_dir = "../reports/figures"
figuras = [f for f in os.listdir(figures_dir) if f.endswith('.png')]
figuras.sort()

print("=== FIGURAS GERADAS PARA O PROJETO ===\n")
for fig_name in figuras:
    path = os.path.join(figures_dir, fig_name)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fig_name:<45} {size_kb:>6.1f} KB")

print(f"\nTotal: {len(figuras)} figuras")
print("\nTodas as visualizacoes estao prontas para incorporar no README.")
print("Use a seguinte sintaxe no README.md:")
print()
for fig_name in figuras:
    print(f"  ![{fig_name.replace('.png','').replace('_',' ')}](reports/figures/{fig_name})")
